In [1]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 69.6 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 103.7 MB/s eta 0:00:0000:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
# # Install the exact libraries used in your backend
!pip install qdrant-client sentence-transformers torch psutil urllib3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 7.7 MB/s eta 0:00:00a 0:00:01


In [3]:
import os
import psutil
import torch
import gc

def print_memory_usage(stage_name):
    """Prints System RAM and GPU VRAM usage."""
    process = psutil.Process(os.getpid())
    sys_ram_mb = process.memory_info().rss / (1024 * 1024)
    
    vram_mb = 0
    if torch.cuda.is_available():
        vram_mb = torch.cuda.memory_allocated() / (1024 * 1024)
        
    print(f"--- Memory Profile: {stage_name} ---")
    print(f"System RAM: {sys_ram_mb:.2f} MB")
    print(f"GPU VRAM:   {vram_mb:.2f} MB\n")

print_memory_usage("Baseline (Libraries Loaded)")

--- Memory Profile: Baseline (Libraries Loaded) ---
System RAM: 572.57 MB
GPU VRAM:   0.00 MB



In [4]:
import urllib.request
import xml.etree.ElementTree as ET

def fetch_sample_papers(category="cs.CL", max_results=50):
    """Simplified version of your ArxivClient."""
    url = f"http://export.arxiv.org/api/query?search_query=cat:{category}&max_results={max_results}"
    
    with urllib.request.urlopen(url) as response:
        xml_data = response.read()
        
    root = ET.fromstring(xml_data)
    ns = {'atom': 'http://www.w3.org/2005/Atom'}
    
    papers = []
    for entry in root.findall('atom:entry', ns):
        arxiv_id = entry.find('atom:id', ns).text.split('/')[-1]
        title = entry.find('atom:title', ns).text.strip().replace('\n', ' ')
        abstract = entry.find('atom:summary', ns).text.strip().replace('\n', ' ')
        papers.append({"arxiv_id": arxiv_id, "title": title, "abstract": abstract, "category": category})
        
    print(f"Fetched {len(papers)} papers from arXiv.")
    return papers

papers = fetch_sample_papers()
print_memory_usage("After Fetching XML Data")

Fetched 50 papers from arXiv.
--- Memory Profile: After Fetching XML Data ---
System RAM: 578.59 MB
GPU VRAM:   0.00 MB



In [5]:
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, VectorParams, Distance

# 1. Load Model onto GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading SentenceTransformer on: {device}")
embedder = SentenceTransformer('all-MiniLM-L6-v2', device=device)

print_memory_usage("After Loading ML Model into VRAM")

# 2. Initialize IN-MEMORY Qdrant (No Docker needed)
qdrant = QdrantClient(location=":memory:")
qdrant.create_collection(
    collection_name="papers",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

# 3. Process and Load
print("Embedding and loading papers into Vector DB...")
points = []
for i, paper in enumerate(papers):
    # Vectorize the abstract
    vector = embedder.encode(paper["abstract"]).tolist()
    
    # Create Qdrant point
    points.append(
        PointStruct(
            id=i, 
            vector=vector, 
            payload={
                "arxiv_id": paper["arxiv_id"], 
                "title": paper["title"], 
                "category": paper["category"],
                "abstract": paper["abstract"] # <--- ADDED THE ABSTRACT HERE
            }
        )
    )

qdrant.upsert(collection_name="papers", points=points)

print_memory_usage("After Populating Vector DB")

Loading SentenceTransformer on: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

--- Memory Profile: After Loading ML Model into VRAM ---
System RAM: 1141.48 MB
GPU VRAM:   86.65 MB

Embedding and loading papers into Vector DB...
--- Memory Profile: After Populating Vector DB ---
System RAM: 1368.36 MB
GPU VRAM:   95.78 MB



In [6]:
test_query = "What are the latest advancements in natural language processing and transformers?"
print(f"Query: '{test_query}'\n")

# Embed the search query
query_vector = embedder.encode(test_query).tolist()

# Search Qdrant using the new Universal Query API
response = qdrant.query_points(
    collection_name="papers",
    query=query_vector,
    limit=3,
    with_payload=True
)

# Extract the list of ScoredPoint objects so it works with our future RAG cell
results = response.points

for r in results:
    print(f"[Score: {r.score:.4f}] {r.payload['title']}")

print("\n")
print_memory_usage("Final Footprint")

Query: 'What are the latest advancements in natural language processing and transformers?'

[Score: 0.3995] I like fish, especially dolphins: Addressing Contradictions in Dialogue Modeling
[Score: 0.3952] Resolving Gendered Ambiguous Pronouns with BERT
[Score: 0.3915] Text2Node: a Cross-Domain System for Mapping Arbitrary Phrases to a Taxonomy


--- Memory Profile: Final Footprint ---
System RAM: 1449.36 MB
GPU VRAM:   95.78 MB



## Local Inference on GPU 
Model page: https://huggingface.co/google/gemma-4-E2B-it

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/google/gemma-4-E2B-it)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [7]:
# Install the Hugging Face ecosystem tools needed for Gemma 4 and 4-bit compression
!pip install accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.7 MB/s eta 0:00:00:00:0100:01


In [8]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

print("Retrieving Hugging Face token from Kaggle Secrets...")
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HFToken")

print("Logging into Hugging Face...")
login(token=hf_token)

Retrieving Hugging Face token from Kaggle Secrets...
Logging into Hugging Face...


In [9]:
import torch
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "google/gemma-4-E2B-it"

print("Configuring 4-bit quantization for low-VRAM environments...")
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

print(f"Loading {MODEL_ID} into GPU memory. This might take a minute...")
processor = AutoProcessor.from_pretrained(MODEL_ID)
llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto" # Automatically places it on your GPU
)

print_memory_usage("After Loading Gemma 4 E2B (4-bit)")

Configuring 4-bit quantization for low-VRAM environments...
Loading google/gemma-4-E2B-it into GPU memory. This might take a minute...


processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

--- Memory Profile: After Loading Gemma 4 E2B (4-bit) ---
System RAM: 2365.10 MB
GPU VRAM:   95.78 MB



In [10]:
def generate_rag_answer(query: str, retrieved_papers: list) -> str:
    """Takes the user query and the database results, and asks Gemma to answer."""
    
    # 1. Format the retrieved papers into a single context block
    context = ""
    for idx, paper in enumerate(retrieved_papers):
        context += f"--- Paper {idx+1}: {paper.payload['title']} ---\n"
        context += f"Abstract: {paper.payload['abstract']}\n\n"

    # 2. Build the System Prompt required by Gemma 4
    system_instruction = (
        "You are a highly capable AI research assistant. "
        "Answer the user's question using ONLY the context provided below from research abstracts. "
        "If the answer is not contained in the context, say 'I cannot answer this based on the provided papers.'\n\n"
        f"CONTEXT:\n{context}"
    )

    # 3. Format the chat template exactly as Gemma 4 expects
    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": query},
    ]

    text_prompt = processor.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True, 
        enable_thinking=False
    )
    
    # 4. Tokenize and push to GPU
    inputs = processor(text=text_prompt, return_tensors="pt").to(llm_model.device)
    input_len = inputs["input_ids"].shape[-1]

    # 5. Generate the raw output
    print("Gemma is reading the papers and writing an answer...\n")
    outputs = llm_model.generate(**inputs, max_new_tokens=512, temperature=0.2)
    
    # 6. Decode and parse output (Official Gemma 4 Syntax)
    response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)
    parsed_output = processor.parse_response(response)
    
    if isinstance(parsed_output, list):
        return "".join([item.get('text', '') for item in parsed_output if 'text' in item])
    elif isinstance(parsed_output, str):
        return parsed_output
    else:
        return str(parsed_output)

# --- TEST THE FULL PIPELINE ---
user_question = "What are the latest advancements in natural language processing and transformers?"

# (Assuming `results` is still in memory from Cell 6)
final_answer = generate_rag_answer(user_question, results)

print("=== FINAL RAG ANSWER ===")
print(final_answer)
print("========================\n")

print_memory_usage("Peak VRAM during Text Generation")

Gemma is reading the papers and writing an answer...

=== FINAL RAG ANSWER ===
{'role': 'assistant', 'content': 'The provided abstracts discuss several specific NLP tasks and models, rather than a broad overview of the latest advancements in NLP and transformers.\n\nThe available information covers:\n*   **Contradiction Detection in Dialogue Modeling:** Introducing the DialoguE COntradiction DEtection task and comparing structured versus unstructured approaches for detecting contradictions in dialogue using pre-trained Transformer models.\n*   **Gendered Pronoun Resolution:** Using a BERT-based approach to solve gender-balanced pronoun resolution, achieving a high F1 score and reducing gender bias.\n*   **Cross-Domain Mapping in Healthcare (Text2Node):** Presenting Text2Node, a cross-domain system for mapping medical phrases to concepts in a taxonomy, focusing on scalability, robustness to wording variants, and generalization.\n\nI cannot provide a general answer about the latest advan